# Teste Mistral API — geração de exemplos N1

Versão Colab baseada diretamente na API HTTP oficial da Mistral.

No painel **Secrets** do Colab, cria `MISTRAL_API_KEY`.

In [1]:
NUMBER_OF_EXAMPLES = 2
OUTPUT_DIR = "/content/mistral_outputs"
OUTPUT_FILE = f"{OUTPUT_DIR}/n1_mistral_test_{NUMBER_OF_EXAMPLES}_examples.json"
DOMAIN = "Bibliotecas"
FACT_TYPE = "Capacidade"
ENTITY_TYPE = "Biblioteca"
CONTEXT_LENGTH = "curto, com 3 a 5 frases"
FACT_POSITION = "no fim do contexto"
QUESTION_TYPE = "pergunta direta"
DISTRACTOR_RULE = "não incluir informação enganadora nem distratores relevantes"
MODEL_CANDIDATES = ["mistral-small-latest", "mistral-small-2603", "ministral-8b-latest", "ministral-3b-latest"]
TEMPERATURE = 0.8
MAX_TOKENS = 4000

In [3]:
import json, re, requests
from pathlib import Path
from google.colab import userdata
api_key = userdata.get("MISTRAL_API_KEY")
if not api_key:
    raise ValueError("Adiciona MISTRAL_API_KEY aos Secrets do Colab.")
BASE_URL = "https://api.mistral.ai/v1"
MODELS_URL = f"{BASE_URL}/models"
CHAT_URL = f"{BASE_URL}/chat/completions"
HEADERS = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Configuração pronta.")

Configuração pronta.


In [4]:
response = requests.get(MODELS_URL, headers=HEADERS, timeout=60)
if not response.ok:
    print(response.status_code)
    print(response.text)
response.raise_for_status()
available_models = sorted(item["id"] for item in response.json().get("data", []) if "id" in item)
print(f"Modelos disponíveis: {len(available_models)}")
for model_id in available_models:
    print(model_id)

Modelos disponíveis: 70
codestral-2508
codestral-embed
codestral-embed-2505
codestral-latest
devstral-2512
devstral-latest
devstral-medium-latest
labs-leanstral-1-5
labs-leanstral-1-5-1
magistral-medium-2509
magistral-medium-latest
magistral-small-2509
magistral-small-latest
ministral-14b-2512
ministral-14b-latest
ministral-3b-2512
ministral-3b-latest
ministral-8b-2512
ministral-8b-latest
mistral-code-agent-latest
mistral-code-fim-latest
mistral-code-latest
mistral-embed
mistral-embed-2312
mistral-large-2512
mistral-large-2512
mistral-large-latest
mistral-large-latest
mistral-medium
mistral-medium
mistral-medium-2505
mistral-medium-2508
mistral-medium-2604
mistral-medium-2604
mistral-medium-3
mistral-medium-3
mistral-medium-3-5
mistral-medium-3-5
mistral-medium-3.5
mistral-medium-3.5
mistral-medium-latest
mistral-medium-latest
mistral-moderation-2603
mistral-ocr-2512
mistral-ocr-3
mistral-ocr-3-0
mistral-ocr-4
mistral-ocr-4-0
mistral-ocr-latest
mistral-small-2506
mistral-small-2603
mis

In [5]:
MODEL_NAME = next((candidate for candidate in MODEL_CANDIDATES if candidate in available_models), None)
if MODEL_NAME is None:
    raise RuntimeError("Nenhum modelo candidato está disponível. Escolhe um nome da lista e adiciona-o a MODEL_CANDIDATES.")
print("Modelo selecionado:", MODEL_NAME)

Modelo selecionado: mistral-small-latest


In [6]:
test_payload = {"model": MODEL_NAME, "messages": [{"role": "user", "content": "Responde apenas com a palavra: Olá"}], "temperature": 0, "max_tokens": 20}
test_response = requests.post(CHAT_URL, headers=HEADERS, json=test_payload, timeout=60)
if not test_response.ok:
    print(test_response.status_code)
    print(test_response.text)
test_response.raise_for_status()
print(test_response.json()["choices"][0]["message"]["content"])

Olá


In [7]:
prompt = f"""
Gera exatamente {NUMBER_OF_EXAMPLES} exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito num contexto fornecido.

DISTRIBUIÇÃO
- domínio: {DOMAIN}
- tipo de facto pedido: {FACT_TYPE}
- tipo de entidade principal: {ENTITY_TYPE}
- contexto: {CONTEXT_LENGTH}
- posição do facto que responde à pergunta: {FACT_POSITION}
- tipo de pergunta: {QUESTION_TYPE}
- distratores: {DISTRACTOR_RULE}

REGRAS
1. Cada exemplo deve conter exatamente context, question e answer.
2. A resposta deve estar claramente escrita no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Usa português europeu.
9. Devolve apenas JSON válido.
10. Não uses Markdown.
11. Não acrescentes explicações.

FORMATO
{{
  "examples": [
    {{
      "context": "...",
      "question": "...",
      "answer": "..."
    }}
  ]
}}
""".strip()
print(prompt)

Gera exatamente 2 exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito num contexto fornecido.

DISTRIBUIÇÃO
- domínio: Bibliotecas
- tipo de facto pedido: Capacidade
- tipo de entidade principal: Biblioteca
- contexto: curto, com 3 a 5 frases
- posição do facto que responde à pergunta: no fim do contexto
- tipo de pergunta: pergunta direta
- distratores: não incluir informação enganadora nem distratores relevantes

REGRAS
1. Cada exemplo deve conter exatamente context, question e answer.
2. A resposta deve estar claramente escrita no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Usa português europeu.
9. Devolve apenas JSON válido.
10. Não uses Markdown.
11. Não acrescentes explicações.

FORMATO
{
  "examples": [
    {
      "context": "...",
      "que

In [8]:
generation_payload = {"model": MODEL_NAME, "messages": [{"role": "system", "content": "Devolve apenas JSON válido. Segue rigorosamente o formato pedido."}, {"role": "user", "content": prompt}], "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS, "response_format": {"type": "json_object"}}
generation_response = requests.post(CHAT_URL, headers=HEADERS, json=generation_payload, timeout=120)
if not generation_response.ok:
    print(generation_response.status_code)
    print(generation_response.text)
generation_response.raise_for_status()
raw_text = generation_response.json()["choices"][0]["message"]["content"].strip()
print(raw_text[:2000])

{"examples": [{"context": "A Biblioteca Municipal de Évora dispõe de espaços dedicados ao estudo em grupo, com capacidade para vinte pessoas em simultâneo. Os utilizadores podem aceder a computadores com ligação à internet e impressoras a cores mediante pedido prévio. O horário de funcionamento estende-se das nove horas às vinte e uma horas, todos os dias úteis.", "question": "Quantas pessoas podem estudar em grupo na Biblioteca Municipal de Évora ao mesmo tempo?", "answer": "Vinte pessoas"}, {"context": "A Biblioteca Municipal de Coimbra oferece um serviço de empréstimo de livros físicos e digitais, com capacidade para cinco mil empréstimos mensais. Os utilizadores registados podem requisitar até dez itens de cada vez, durante um período de quinze dias. O balcão de atendimento está aberto de segunda a sexta-feira, das dez horas às dezanove horas.", "question": "Quantos empréstimos mensais a Biblioteca Municipal de Coimbra consegue gerir?", "answer": "Cinco mil empréstimos"}]}


In [9]:
def clean_json_text(text):
    text = text.strip()
    if text.startswith("```json"):
        text = text[len("```json"):]
    if text.startswith("```"):
        text = text[len("```"):]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()
payload = json.loads(clean_json_text(raw_text))
if not isinstance(payload, dict) or "examples" not in payload:
    raise ValueError("A resposta deve ser um objeto com a chave 'examples'.")
examples = payload["examples"]
if len(examples) != NUMBER_OF_EXAMPLES:
    raise ValueError(f"Esperava {NUMBER_OF_EXAMPLES} exemplos, recebi {len(examples)}.")
required_fields = {"context", "question", "answer"}
for index, example in enumerate(examples, start=1):
    missing = required_fields - set(example.keys())
    if missing:
        raise ValueError(f"Exemplo {index}: faltam os campos {sorted(missing)}.")
print(f"JSON válido: {len(examples)} exemplos.")

JSON válido: 2 exemplos.


In [10]:
def normalize(text):
    text = re.sub(r"\s+", " ", str(text).lower().strip())
    return text.strip(" .,:;!?")
for index, example in enumerate(examples, start=1):
    answer_in_context = normalize(example["answer"]) in normalize(example["context"])
    print(f"Exemplo {index}: resposta no contexto = {answer_in_context}")

Exemplo 1: resposta no contexto = True
Exemplo 2: resposta no contexto = True


In [11]:
for index, example in enumerate(examples, start=1):
    print("\n" + "=" * 80)
    print("EXEMPLO", index)
    print("Contexto:", example["context"])
    print("Pergunta:", example["question"])
    print("Resposta:", example["answer"])


EXEMPLO 1
Contexto: A Biblioteca Municipal de Évora dispõe de espaços dedicados ao estudo em grupo, com capacidade para vinte pessoas em simultâneo. Os utilizadores podem aceder a computadores com ligação à internet e impressoras a cores mediante pedido prévio. O horário de funcionamento estende-se das nove horas às vinte e uma horas, todos os dias úteis.
Pergunta: Quantas pessoas podem estudar em grupo na Biblioteca Municipal de Évora ao mesmo tempo?
Resposta: Vinte pessoas

EXEMPLO 2
Contexto: A Biblioteca Municipal de Coimbra oferece um serviço de empréstimo de livros físicos e digitais, com capacidade para cinco mil empréstimos mensais. Os utilizadores registados podem requisitar até dez itens de cada vez, durante um período de quinze dias. O balcão de atendimento está aberto de segunda a sexta-feira, das dez horas às dezanove horas.
Pergunta: Quantos empréstimos mensais a Biblioteca Municipal de Coimbra consegue gerir?
Resposta: Cinco mil empréstimos


In [12]:
result = {"provider": "mistral", "model": MODEL_NAME, "number_of_examples": len(examples), "generation_settings": {"temperature": TEMPERATURE, "domain": DOMAIN, "fact_type": FACT_TYPE, "entity_type": ENTITY_TYPE, "context_length": CONTEXT_LENGTH, "fact_position": FACT_POSITION, "question_type": QUESTION_TYPE}, "examples": examples}
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)
print("Guardado em:", OUTPUT_FILE)

Guardado em: /content/mistral_outputs/n1_mistral_test_2_examples.json


In [14]:
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Depois de validares os 2 exemplos, altera `NUMBER_OF_EXAMPLES = 2` para `20`.

Este notebook valida apenas a Mistral, mantendo o mesmo protocolo usado para Gemini e Groq.